In [9]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score

PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.features import *
from src.volatility import *
from src.labeling import *

In [10]:
df = pd.read_csv("../data/raw/nifty50.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df = add_returns(df)
df = add_close_to_close_volatility(df, window=20)

In [11]:
hmm_df = pd.read_csv("../data/processed/hmm_regimes.csv", parse_dates=["Date"])

df = df.merge(
    hmm_df,
    on="Date",
    how="inner"
)

df = df.dropna().reset_index(drop=True)

In [12]:
df = add_target_direction(df)
df = add_lagged_returns(df, lags=(1,5,10))
df = add_moving_average_features(df, windows=(5,10,20))
df = add_volatility_features(df, vol_col="vol_cc", lags=(1,))

df = df.dropna().reset_index(drop=True)

In [13]:
regime_dummies = pd.get_dummies(
    df["hmm_regime"],
    prefix="regime"
)

df = pd.concat([df, regime_dummies], axis=1)

df.head()

,Date,Open,High,Low,Close,Volume,return,vol_cc,hmm_regime,target,ret_lag_1,ret_lag_5,ret_lag_10,ma_ratio_5,ma_ratio_10,ma_ratio_20,vol_cc_lag_1,regime_High,regime_Low,regime_Medium
0,2007-11-13,5612.350098,5758.850098,5591.600098,5695.399902,0,0.013940,0.338223,High,1,-0.014328,-0.014345,0.035705,0.996393,0.982039,1.006963,0.369259,True,False,False
1,2007-11-14,5703.950195,5950.200195,5700.049805,5937.899902,0,0.042578,0.368823,High,0,0.013940,-0.010398,-0.006290,1.033343,1.022633,1.047340,0.338223,True,False,False
2,2007-11-15,5942.700195,5966.950195,5895.649902,5912.100098,0,-0.004345,0.360871,High,0,0.042578,-0.000717,0.005436,1.024228,1.017989,1.039555,0.368823,True,False,False
3,2007-11-16,5913.149902,5948.049805,5817.399902,5906.850098,0,-0.000888,0.327000,High,1,-0.004345,-0.014458,-0.005796,1.015993,1.016378,1.033580,0.360871,True,False,False
4,2007-11-19,5908.049805,5981.799805,5893.799805,5907.649902,0,0.000135,0.307166,High,0,-0.000888,-0.014328,0.011242,1.006075,1.016949,1.027496,0.327000,True,False,False


In [15]:
feature_cols = [
    "ret_lag_1","ret_lag_5","ret_lag_10",
    "ma_ratio_5","ma_ratio_10","ma_ratio_20",
    "vol_cc","vol_cc_lag_1"
]
feature_cols_regime = feature_cols + list(regime_dummies.columns)

In [16]:
split = int(len(df)*0.8)

train = df.iloc[:split].copy()
test  = df.iloc[split:].copy()

y_train = train["target"]
y_test  = test["target"]

In [17]:
scaler = StandardScaler()

X_train = scaler.fit_transform(train[feature_cols])
X_test  = scaler.transform(test[feature_cols])

logit = LogisticRegression(max_iter=1000)
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1
)
gb = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    random_state=42
)

logit.fit(X_train, y_train)
rf.fit(X_train, y_train)
gb.fit(X_train, y_train)

pred_logit = logit.predict(X_test)
pred_rf = rf.predict(X_test)
pred_gb = gb.predict(X_test)

In [18]:
scaler_r = StandardScaler()

X_train_r = scaler_r.fit_transform(train[feature_cols_regime])
X_test_r  = scaler_r.transform(test[feature_cols_regime])

logit_r = LogisticRegression(max_iter=1000)
rf_r = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1
)
gb_r = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    random_state=42
)

logit_r.fit(X_train_r, y_train)
rf_r.fit(X_train_r, y_train)
gb_r.fit(X_train_r, y_train)

pred_logit_r = logit_r.predict(X_test_r)
pred_rf_r = rf_r.predict(X_test_r)
pred_gb_r = gb_r.predict(X_test_r)

In [19]:
results_regime_feature = pd.DataFrame({
    "Model": ["Logistic","RandomForest","GradientBoost"],
    "Baseline": [
        accuracy_score(y_test, pred_logit),
        accuracy_score(y_test, pred_rf),
        accuracy_score(y_test, pred_gb)
    ],
    "With_HMM_Regime": [
        accuracy_score(y_test, pred_logit_r),
        accuracy_score(y_test, pred_rf_r),
        accuracy_score(y_test, pred_gb_r)
    ]
})

results_regime_feature["Gain"] = (
    results_regime_feature["With_HMM_Regime"]
    - results_regime_feature["Baseline"]
)

results_regime_feature

,Model,Baseline,With_HMM_Regime,Gain
0,Logistic,0.550725,0.554069,0.003344
1,RandomForest,0.559643,0.549610,-0.010033
2,GradientBoost,0.556299,0.561873,0.005574
